# KnowledgeHub RAG v0.6 – Enhanced Retrieval-Augmented Generation

## Objective

The objective of v0.6 is to enhance the Retrieval-Augmented Generation (RAG) system by improving both document retrieval and answer generation. This version combines an enhanced hybrid retrieval pipeline with a lightweight instruction-tuned Large Language Model (TinyLlama) to generate accurate, context-grounded responses from uploaded documents.

Compared to previous versions, v0.6 introduces query expansion, weighted score fusion, Reciprocal Rank Fusion (RRF), and context expansion to improve retrieval quality before passing the retrieved information to the language model for answer generation.

## New Features

- Integrates semantic search (FAISS) and lexical search (BM25) into a hybrid retrieval pipeline.
- Improves retrieval using weighted score fusion before Reciprocal Rank Fusion (RRF).
- Introduces lightweight query expansion to improve retrieval for abbreviated and technical queries.
- Expands retrieved context using neighbouring document chunks to preserve surrounding information.
- Generates natural language answers using the TinyLlama instruction-tuned language model.
- Grounds responses strictly on the retrieved document context to reduce hallucinations.
- Provides an interactive question-answering interface for uploaded PDF documents.

## Workflow

User Question
↓
Query Expansion
↓
Semantic Search (FAISS) + BM25 Retrieval
↓
Weighted Score Fusion
↓
Reciprocal Rank Fusion (RRF)
↓
Context Expansion
↓
TinyLlama LLM
↓
Grounded Natural Language Answer

## Learning Objectives

- Understand the architecture of Retrieval-Augmented Generation (RAG) systems.
- Combine semantic and lexical retrieval techniques for improved document retrieval.
- Apply weighted score fusion and Reciprocal Rank Fusion to improve retrieval ranking.
- Improve retrieval robustness through query expansion and context expansion.
- Integrate an instruction-tuned Large Language Model for grounded answer generation.
- Build an end-to-end document question-answering system capable of interactive conversations.

## Version Highlights

- Hybrid retrieval (FAISS + BM25)
- Query expansion
- Weighted score fusion
- Reciprocal Rank Fusion (RRF)
- Context expansion
- TinyLlama-powered answer generation
- Context-grounded responses
- Interactive document question-answering interface

In [56]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate
!pip install -q rank-bm25
!pip install -q langchain

In [14]:
import os
import time
import faiss
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from rank_bm25 import BM25Okapi

In [58]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 1 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf


In [16]:
def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        extracted = page.extract_text()

        if extracted:

            pages.append(
                {
                    "page": page_number,
                    "text": extracted
                }
            )

    return pages


documents = []

for pdf in pdf_files:

    pages = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "pages": pages
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 1 document(s).


In [17]:
!pip install -q langchain-text-splitters

In [18]:
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================================
# Heading Detection
# ==========================================================

def is_heading(line):

    line = line.strip()

    if len(line) < 2:
        return False

    # 1
    # 1.2
    # 2.3.4
    if re.match(r"^\d+(\.\d+)*\s+[A-Z]", line):
        return True

    # ALL CAPS
    if line.isupper() and len(line.split()) <= 8:
        return True

    # Markdown
    if line.startswith("#"):
        return True

    # Very short title
    if len(line.split()) <= 8 and line.endswith(":"):
        return True

    return False


# ==========================================================
# Split page into sections using headings
# ==========================================================

def split_into_sections(text):

    lines = text.split("\n")

    sections = []

    current = []

    for line in lines:

        if is_heading(line):

            if current:
                sections.append("\n".join(current).strip())

            current = [line]

        else:

            current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    return sections


# ==========================================================
# Recursive splitter
# ==========================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


# ==========================================================
# Final Chunking Function
# ==========================================================

def chunk_text(text):

    sections = split_into_sections(text)

    final_chunks = []

    for section in sections:

        if len(section) <= 900:

            final_chunks.append(section)

        else:

            final_chunks.extend(
                splitter.split_text(section)
            )

    return final_chunks

In [19]:
all_chunks = []

chunk_id = 0

for document in documents:

    for page in document["pages"]:

        sections = split_into_sections(page["text"])

        for section in sections:

            # -----------------------------
            # Extract heading
            # -----------------------------
            lines = section.split("\n")

            heading = ""

            if lines and is_heading(lines[0]):
                heading = lines[0].strip()

            # -----------------------------
            # Split section if needed
            # -----------------------------
            if len(section) <= 900:

                section_chunks = [section]

            else:

                section_chunks = splitter.split_text(section)

            # -----------------------------
            # Save chunks
            # -----------------------------
            for chunk in section_chunks:

                all_chunks.append(
                    {
                        "chunk_id": chunk_id,
                        "document": document["filename"],
                        "page": page["page"],
                        "section": heading,
                        "text": chunk
                    }
                )

                chunk_id += 1

print(f"Created {len(all_chunks)} chunks.")

Created 112 chunks.


In [20]:
print("=" * 80)

print("FIRST 10 CHUNKS")

print("=" * 80)

for chunk in all_chunks[:10]:

    print(f"\nChunk ID : {chunk['chunk_id']}")

    print(f"Page     : {chunk['page']}")


    print("-" * 80)

    print(chunk["text"][:400])

    print()

FIRST 10 CHUNKS

Chunk ID : 0
Page     : 1
--------------------------------------------------------------------------------
Machine Learning Classification of Binary Neutron
Star Remnants Using Gravitational Wave Data
Surendaranath Kanniyappan
Dr. Michalis Agathos
Abstract
Binary neutron star (BNS) mergers are among the most energetic cosmic events,
producing gravitational waves (GWs), electromagnetic (EM) counterparts, and potentially
neutrinos. These mergers provide an unparalleled opportunity to study supranuclear m


Chunk ID : 1
Page     : 1
--------------------------------------------------------------------------------
hypermassive neutron star (short- or long-lived HMNS), or remains stable.
Direct detection of postmerger GW signals remains challenging due to their high
frequency nature (≳ 1 kHz) and the sensitivity limits of current interferometers. Therefore,
predicting remnant outcomes from inspiral parameters—total mass Mtot, mass ratio q, tidal
deformability ˜Λ, and effecti

In [21]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(112, 384)


In [23]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

112


In [24]:

tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print(" BM25 Index Built")

 BM25 Index Built


In [25]:
QUERY_EXPANSION = {

    "algorithm": [
        "model",
        "classifier",
        "GBDT",
        "Gradient Boosted Decision Tree"
    ],

    "gbdt": [
        "Gradient Boosted Decision Tree",
        "gradient boosting"
    ],

    "classifier": [
        "classification model",
        "machine learning model"
    ],

    "accuracy": [
        "performance",
        "evaluation",
        "MCC"
    ],

    "dataset": [
        "training data",
        "simulation dataset"
    ],

    "method": [
        "approach",
        "framework"
    ]
}


def expand_query(query):

    expanded = query

    query_lower = query.lower()

    for key, values in QUERY_EXPANSION.items():

        if key in query_lower:

            expanded += " " + " ".join(values)

    return expanded

In [26]:
def reciprocal_rank_fusion(semantic_results, bm25_results, k=60):
    """
    Reciprocal Rank Fusion (RRF)

    Score(doc) = Σ 1 / (k + rank)

    Larger k -> smoother scores (60 is the standard value).
    """

    fused = {}

    # Semantic ranking
    for rank, item in enumerate(semantic_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    # BM25 ranking
    for rank, item in enumerate(bm25_results, start=1):
        cid = item["chunk_id"]

        if cid not in fused:
            fused[cid] = item.copy()
            fused[cid]["rrf_score"] = 0.0

        fused[cid]["rrf_score"] += 1.0 / (k + rank)

    return sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

In [27]:
def retrieve(query, top_k=5):

    # =====================================================
    # Encode Query
    # =====================================================
    expanded_query = expand_query(query)

    query_embedding = embedding_model.encode(
        [expanded_query],
        normalize_embeddings=True
    ).astype("float32")

    # =====================================================
    # FAISS Search
    # =====================================================

    semantic_scores, semantic_indices = index.search(
        query_embedding,
        top_k * 8
    )

    semantic_results = []

    for rank, (score, idx) in enumerate(
        zip(semantic_scores[0], semantic_indices[0]),
        start=1
    ):

        semantic_results.append({

            "chunk_id": idx,
            "rank": rank,
            "semantic_score": float(score)

        })

    # =====================================================
    # BM25 Search
    # =====================================================

    tokenized_query = expanded_query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_ranked = sorted(

        enumerate(bm25_scores),

        key=lambda x: x[1],

        reverse=True

    )[:top_k * 8]

    bm25_results = []

    for rank, (idx, score) in enumerate(
        bm25_ranked,
        start=1
    ):

        bm25_results.append({

            "chunk_id": idx,
            "rank": rank,
            "bm25_score": float(score)

        })

    # =====================================================
    # Reciprocal Rank Fusion (RRF)
    # =====================================================

    k = 60

    rrf_scores = {}

    semantic_lookup = {}
    bm25_lookup = {}

    for item in semantic_results:

        cid = item["chunk_id"]

        semantic_lookup[cid] = item["semantic_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    for item in bm25_results:

        cid = item["chunk_id"]

        bm25_lookup[cid] = item["bm25_score"]

        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + item["rank"])

    rrf_results = []

    for chunk_id, rrf_score in rrf_scores.items():

        chunk = all_chunks[chunk_id]

        rrf_results.append({

            "chunk_id": chunk_id,

            "page": chunk["page"],

            "document": chunk["document"],

            "text": chunk["text"],

            "semantic_score": semantic_lookup.get(chunk_id, 0.0),

            "bm25_score": bm25_lookup.get(chunk_id, 0.0),

            "rrf_score": rrf_score

        })

    # =====================================================
    # Intelligent Tie Breaking
    # =====================================================

    for item in rrf_results:

        item["final_score"] = (

            item["rrf_score"]

            + 0.001 * item["semantic_score"]

            + 0.001 * item["bm25_score"]

        )

    rrf_results = sorted(

        rrf_results,

        key=lambda x: x["final_score"],

        reverse=True

    )

    # =====================================================
    # Prevent Adjacent Matched Chunks
    # =====================================================

    selected = []

    for item in rrf_results:

        current = item["chunk_id"]

        if any(abs(current - x["chunk_id"]) <= 1 for x in selected):
            continue

        selected.append(item)

        if len(selected) == top_k:
            break

    # =====================================================
    # Context Expansion
    # =====================================================

    expanded = []

    visited = set()

    for rank, item in enumerate(selected, start=1):

        current = item["chunk_id"]

        for neighbour in [current - 1, current, current + 1]:

            if neighbour < 0:
                continue

            if neighbour >= len(all_chunks):
                continue

            if neighbour in visited:
                continue

            if all_chunks[neighbour]["document"] != item["document"]:
                continue

            visited.add(neighbour)

            chunk = all_chunks[neighbour]

            expanded.append({

                "chunk_id": chunk["chunk_id"],

                "page": chunk["page"],

                "document": chunk["document"],

                "text": chunk["text"],

                "retrieval_rank": rank,

                "context_neighbor": neighbour != current,

                "semantic_score": item["semantic_score"] if neighbour == current else None,

                "bm25_score": item["bm25_score"] if neighbour == current else None,

                "rrf_score": item["rrf_score"] if neighbour == current else None

            })

    return expanded

In [28]:
def debug_retrieval(query, top_k=5):

    results = retrieve(query, top_k)

    print("=" * 90)
    print(f"QUERY : {query}")
    print("=" * 90)

    current_rank = None

    for chunk in results:

        if chunk["retrieval_rank"] != current_rank:

            current_rank = chunk["retrieval_rank"]

            print()
            print("=" * 90)
            print(f"RETRIEVAL RANK {current_rank}")
            print("=" * 90)

        print()

        if chunk["context_neighbor"]:

            print("Context Chunk")

        else:

            print("Matched Chunk")

            print(f"Semantic Score : {chunk['semantic_score']:.4f}")
            print(f"BM25 Score     : {chunk['bm25_score']:.4f}")
            if "rrf_score" in chunk:
              print(f"RRF Score      : {chunk['rrf_score']:.5f}")

        print(f"Document : {chunk['document']}")
        print(f"Page     : {chunk['page']}")

        if chunk.get("section"):
          print(f"Section  : {chunk['section']}")

        print(f"Chunk ID : {chunk['chunk_id']}")

        print("-" * 90)

        print(chunk["text"][:700])

        print()

In [29]:
debug_retrieval("What algorithm was used for classification?")

debug_retrieval("What is GBDT?")

debug_retrieval("Gradient Boosted Decision Tree")

QUERY : What algorithm was used for classification?

RETRIEVAL RANK 1

Context Chunk
Document : Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Page     : 8
Chunk ID : 33
------------------------------------------------------------------------------------------
difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.


Matched Chunk
Semantic Score : 0.6347
BM25 Score     : 13.7764
RRF Score      : 0.03

In [30]:
query = "What algorithm was used for classification?"

results = retrieve(query)

# Keep only the best 2 chunks
context = "\n\n".join(
    [r["text"][:600] for r in results[:2]]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more reali

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs are combined, weighted by a learning r

In [31]:
query = "What algorithm was used for classification?"

results = retrieve(query)

print(f"Retrieved {len(results)} chunks.")

Retrieved 15 chunks.


In [32]:
TOP_CONTEXT = 3

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:TOP_CONTEXT]
)

print("=" * 80)
print(context)

difficulty-aware splitting strategy described in [11] is adopted:
1. Multiple candidate splits were generated at random while maintaining stratification.
2. Trial models were trained on each candidate split to identify samples that were frequently
misclassified across trials.
3. Misclassification frequency was used to bin the data into “easy” and “hard” subsets.
4. Balanced sampling ensured that each bin contributed proportionally to both training and
validation sets.
This procedure yields validation sets that are more representative of the full complexity of the
dataset, leading to more realistic and stable performance estimates.

3.5 Gradient Boosted Decision Tree (GBDT) Training
All three classifiers were trained using the gradient boosting framework for decision trees [27]
as implemented in scikit-learn [28]. This method builds an ensemble of shallow decision
trees (weak learners) sequentially, with each new tree correcting the residual errors of the
previous ensemble. The outputs 

In [33]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("TinyLlama Loaded")

Loading tokenizer...
Loading model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TinyLlama Loaded


In [34]:
def build_prompt(question, context):

    return f"""
You are an AI assistant answering questions about a research report.

You MUST follow these rules:

1. Answer ONLY using the provided context.
2. Do NOT use outside knowledge.
3. Do NOT invent names, facts, numbers or explanations.
4. Keep your answer under three sentences.
5. Do NOT repeat the question.
6. Do NOT repeat the context.
7. Output ONLY the final answer.

Context:
{context}

Question:
{question}

Answer:
"""

In [35]:
def generate_answer(question, context):

    prompt = build_prompt(question, context)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
    **inputs,
    do_sample=False,
    repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [37]:
query = "What algorithm was used for classification?"

results = retrieve(query)

context = "\n\n".join(
    chunk["text"]
    for chunk in results[:3]
)

answer = generate_answer(query, context)

print("=" * 80)
print("QUESTION")
print(query)

print("\n" + "=" * 80)
print("ANSWER")
print(answer)

QUESTION
What algorithm was used for classification?

ANSWER
The algorithm used for classification was Gradient Boosted Decision Trees (GBDT).


In [40]:
while True:
    print("\n" + "=" * 80)

    question = input("\nAsk a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("\nGoodbye!")
        break

    results = retrieve(question)

    context = "\n\n".join(
        chunk["text"]
        for chunk in results[:4]
    )

    answer = generate_answer(
    question,
    context
    )

    print("\n" + "=" * 80)
    print(answer)



Ask a question (type 'exit' to quit): What algorithm was used for classification?

The classification algorithm used was GBDT.


Ask a question (type 'exit' to quit): How was the dataset split?

The dataset was divided into a training set (90% of the data) and a validation set (10%). The
training set was used to train the classifiers, while the validation set was used to evaluate
their performance. The difficulty-aware splitting strategy was used to generate multiple candidate
splits, with misclassification frequencies being used to bin the data into “easy” and “hard”
subsets. The balanced sampling method was employed to ensure that each bin contributed
proportionally to both training and validation sets.


Ask a question (type 'exit' to quit): What is SHAP analysis?

SHAP (Shapley Additive Explanations) is a technique used to understand how a model's output
relates to its input features. It provides a way to visualize the contribution of each feature
to the predicted outcome, allowi